# Apply Live Segmentation Review

Convert reviewed webcam/live mask decisions into a supervised adaptation dataset for fine-tuning.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from app.ml.live_segmentation import LIVE_SEGMENTATION_DATASET_DIR, apply_live_segmentation_review

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
DATASET_DIR = LIVE_SEGMENTATION_DATASET_DIR
REVIEW_CSV_PATH = DATASET_DIR / 'review' / 'live_segmentation_review_template.csv'
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
SEED = 42

print('Dataset dir:', DATASET_DIR)
print('Review CSV:', REVIEW_CSV_PATH)

Dataset dir: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation
Review CSV: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\review\live_segmentation_review_template.csv


In [3]:
export_paths = apply_live_segmentation_review(
    REVIEW_CSV_PATH,
    DATASET_DIR,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=SEED,
)

for key, value in export_paths.items():
    print(f'{key}: {value}')

print('\nReview summary:')
print(export_paths['review_summary'].read_text(encoding='utf-8'))
print('\nSplit summary:')
print(export_paths['summary'].read_text(encoding='utf-8'))

train: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\train.jsonl
val: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\val.jsonl
test: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\test.jsonl
summary: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\split_summary.json
approved: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\reviewed\approved_records.jsonl
rejected: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\reviewed\rejected_records.jsonl
review_summary: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\datasets\live_segmentation_adaptation\reviewed\review_summary

In [4]:
approved_df = pd.read_json(export_paths['approved'], lines=True) if export_paths['approved'].exists() else pd.DataFrame()
rejected_df = pd.read_json(export_paths['rejected'], lines=True) if export_paths['rejected'].exists() else pd.DataFrame()

print('Approved rows:', len(approved_df))
print('Rejected rows:', len(rejected_df))
if not approved_df.empty:
    display(approved_df.head(12))

Approved rows: 0
Rejected rows: 31


## Next step

After this produces non-empty train/val/test manifests, run `notebooks/05_model_training/05_finetune_live_segmentation_model.ipynb`.